# TFM Deteccion - Ejecucion final

Pipeline definitivo: solo CNNDetection, 3 modelos (ResNet-50, ViT-B/16, UFD; FreqNet en stand-by), trainset de ejecucion ~10-12k imgs por muestreo proporcional sobre las 20 categorias de progan_train. Validacion (progan_val) y test (progan_test + CNN_synth_testset, E1+E1b) completos.

Requisitos:
- Runtime Colab con GPU (T4 o superior).
- En `MyDrive/cnndetection-datasets/`: `progan_train.zip`, `progan_val.zip`, `progan_testset.zip`, `CNN_synth_testset.zip`.
- (Opcional) cuenta de wandb.

## 1. Drive + repo + dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -rf /content/tfm_deteccion
!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git
%cd /content/tfm_deteccion
!git pull

In [ ]:
!pip install -q -r requirements.txt

## 2. Inventario del progan_train.zip

Lee el indice del zip directamente desde Drive (sin descomprimir nada), para ver categorias, conteos real/fake y tamanos. Util para decidir el `--total` del trainset de ejecucion.

In [ ]:
ZIP_TRAIN = '/content/drive/MyDrive/cnndetection-datasets/progan_train.zip'
!python scripts/scan_cnndetection.py --zip {ZIP_TRAIN} --csv reports/scan_progan_train.csv

## 3. Construccion del trainset de ejecucion

Muestreo proporcional al tamano de cada categoria, con 50/50 real-fake dentro de cada una. La misma particion (semilla fija) se usara para los 3 modelos.

In [ ]:
TARGET_TOTAL = 12000   # tope objetivo, ajustable tras ver el inventario
SEED = 42
OUT_TRAIN = '/content/cnndetection/progan_train'

!python scripts/build_trainset_ejecucion.py \
    --zip {ZIP_TRAIN} \
    --out {OUT_TRAIN} \
    --total {TARGET_TOTAL} \
    --seed {SEED} \
    --manifest reports/trainset_ejecucion_manifest.json

## 4. Extraccion de val + tests (completos)

progan_val (validacion durante el entrenamiento), progan_testset (E1) y CNN_synth_testset (E1b).

In [ ]:
import os, subprocess

DRIVE_ZIPS = '/content/drive/MyDrive/cnndetection-datasets'
CNN_ROOT = '/content/cnndetection'

for zip_name, dst in [
    ('progan_val.zip',        f'{CNN_ROOT}/progan_val'),
    ('progan_testset.zip',    CNN_ROOT),
    ('CNN_synth_testset.zip', CNN_ROOT),
]:
    print(f'extrayendo {zip_name}...')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['unzip', '-q', '-n', f'{DRIVE_ZIPS}/{zip_name}', '-d', dst], check=True)

if os.path.isdir(f'{CNN_ROOT}/progan_testset') and not os.path.isdir(f'{CNN_ROOT}/progan_test'):
    os.rename(f'{CNN_ROOT}/progan_testset', f'{CNN_ROOT}/progan_test')
    print('renombrado progan_testset -> progan_test')

print('\nEstructura:')
!ls {CNN_ROOT}
!df -h /content | grep -v Filesystem

## 5. Patch de base.yaml

Apunta los roots a `/content/cnndetection`, habilita E1+E1b, deja E2 desactivado y guarda checkpoints en Drive para sobrevivir caidas de sesion.

In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

cfg['data']['train']['root']       = '/content/cnndetection'
cfg['data']['val']['root']         = '/content/cnndetection'
cfg['data']['eval_e1']['root']     = '/content/cnndetection'
cfg['data']['eval_e1']['split']    = 'test'
cfg['data']['eval_e1b']['root']    = '/content/cnndetection'
cfg['data']['eval_e1b']['enabled'] = True
cfg['data']['eval_e2']['enabled']  = False
cfg['data']['num_workers']         = 2

cfg['training']['epochs']     = 8
cfg['training']['batch_size'] = 128
cfg['training']['scheduler']['T_max'] = 8

cfg['output']['base_dir'] = '/content/drive/MyDrive/tfm-checkpoints'
cfg['wandb']['enabled'] = True

cfg_path.write_text(yaml.dump(cfg))
print(yaml.dump(cfg))

## 6. wandb (opcional)

Si no quieres tracking, salta esta celda y pon `cfg['wandb']['enabled'] = False` arriba.

In [ ]:
!wandb login

## 7. Sanity check

In [ ]:
import sys
sys.path.insert(0, '/content/tfm_deteccion')
from data.cnndetection_dataset import CNNDetectionDataset
from data.transforms import get_eval_transforms

for split in ('train', 'val', 'test'):
    try:
        ds = CNNDetectionDataset(root='/content/cnndetection', split=split, transform=get_eval_transforms())
        print(f'{split}: {len(ds)} muestras, generadores={ds.generators}')
    except Exception as e:
        print(f'{split}: ERROR - {e}')

## 8. Train + evaluate de los 3 modelos

`scripts/run_all.py` ya filtra FreqNet (stand-by). Si Colab se cae a mitad, los checkpoints estan en Drive; relanzar con `--eval-only` para retomar solo la evaluacion.

In [ ]:
!python scripts/run_all.py

### Alternativa: modelo a modelo

In [ ]:
modelo = 'resnet50'   # resnet50 | vit | universalfakedetect
!python scripts/train.py --config configs/{modelo}.yaml

In [ ]:
import glob

modelo = 'resnet50'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"

## 9. Resumen de metricas

In [ ]:
import glob, json
from pathlib import Path

rows = []
for modelo in ['resnet50', 'vit', 'universalfakedetect']:
    runs = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*'))
    if not runs:
        print(f'{modelo}: sin runs')
        continue
    last = Path(runs[-1])
    mpath = last / 'metrics.json'
    if not mpath.exists():
        print(f'{modelo}: {last.name} (sin metrics.json)')
        continue
    m = json.loads(mpath.read_text())
    for ronda, vals in m.items():
        rows.append((modelo, ronda, vals['auc_roc'], vals['average_precision'], vals['accuracy']))
        print(f"{modelo:<22} {ronda:<8} AUC={vals['auc_roc']:.4f} AP={vals['average_precision']:.4f} Acc={vals['accuracy']:.4f}")

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=['modelo', 'ronda', 'AUC', 'AP', 'Acc'])
    df.to_csv('/content/drive/MyDrive/tfm-checkpoints/resumen_metricas.csv', index=False)
    print('\nCSV guardado en Drive: tfm-checkpoints/resumen_metricas.csv')
except Exception as e:
    print('No se pudo guardar CSV:', e)